# 07 · Explainable AI: what the models learned

Two complementary explanations, both computed on **held-out data**: the outer test folds of the primary block size, with the outer-fold models saved by notebook 03. Each test cell is explained by a model that never saw its borehole.

| Method | Question it answers | How |
|---|---|---|
| **Permutation importance** (Breiman 2001; Fisher et al. 2019) | How much skill is lost if an input is made useless? | Shuffle one input (or a group) across the test cells of a fold and measure the drop in balanced accuracy; 10 repeats per fold; mean ± SD over the 5 folds |
| **SHAP** (Lundberg & Lee 2017) | How does each input push an individual prediction towards Clay or Sand? | Exact Shapley values of P(Clay) for the 3 inputs, with a background sample of the fold's training cells; the same model-agnostic explainer for every model, so the models are comparable |

**Grouped importance.** Easting and Northing together are *location*; `Top_Elevation_m` (ground elevation − depth) carries the depth trend. Shuffling a group answers the geotechnical question directly: how much of the skill comes from where a cell is, and how much from how deep it is. Inputs that carry related information share credit, so single-input importances should be read together with the grouped ones.

**Measured cells only.** Every importance is computed on all test cells and repeated on measured cells only, as in notebook 04.

Figures without coordinates go to `outputs/07_xai/aggregate/`; the cell-level SHAP values (with borehole codes) go to `restricted/`.

In [ ]:
import os, sys, json
from pathlib import Path

# Run mode: "full" for the experiment, "smoke" for the reduced software validation run.
# os.environ["SIEMREAP_MODE"] = "smoke"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "project.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

from siemreap import config
cfg = config.load_config(ROOT)
MODE = config.run_mode()
print(config.smoke_banner())

import joblib, warnings
from sklearn.metrics import balanced_accuracy_score
from siemreap import folds as F, plots
from siemreap.data import FEATURES
AGG = config.stage_dir(cfg, "07_xai", "aggregate")
RES = config.stage_dir(cfg, "07_xai", "restricted")
MOD = config.stage_dir(cfg, "03_nested_cv", "") / "models"
mt = pd.read_csv(config.processed_dir(cfg) / "cells_all.csv")
preds = pd.read_csv(config.stage_dir(cfg, "03_nested_cv", "restricted") / "outer_predictions.csv")
primary = int(preds.block_m.mode()[0]) if MODE == "smoke" else F.primary_block(cfg)
outer = F.load_outer(cfg, primary)
models = [m for m in cfg["experiment"]["models"] if m in set(preds.model)]
folds_k = sorted(preds[(preds.block_m == primary) & (preds.purge_m == 0)].outer_fold.unique())
NAMES = {"UTM_Easting_m": "Easting", "UTM_Northing_m": "Northing", "Top_Elevation_m": "Elevation"}
GROUPS = {"Location (Easting + Northing)": ["UTM_Easting_m", "UTM_Northing_m"], "Elevation (depth trend)": ["Top_Elevation_m"]}
REPEATS = 3 if MODE == "smoke" else 10
print("primary block:", primary, "m | models:", models, "| folds:", folds_k)

def load_fold(model, k):
    tag = f"A_{primary}m_p0_{model}"
    m = joblib.load(MOD / f"{tag}_fold{k}.joblib")
    thr = float(preds[(preds.model == model) & (preds.block_m == primary) & (preds.purge_m == 0)
                      & (preds.outer_fold == k)].threshold.iloc[0])
    train, test = F.split(mt, outer, k, "A")
    if not cfg["depth"].get("score_filled"):
        test = test[test.measured == 1]
    return m, thr, train, test

## 1. Held-out permutation importance

In [ ]:
rng = np.random.default_rng(cfg["seeds"]["global"])
rows = []
for model in models:
    for k in folds_k:
        m, thr, _, test = load_fold(model, k)
        X0, y = test[FEATURES].values, test.y.values
        meas = test.measured.values == 1
        def score(X, mask):
            p = m.predict_proba(X)[:, 1]
            return balanced_accuracy_score(y[mask], (p[mask] >= thr).astype(int))
        for subset, mask in (("all cells", np.ones(len(y), bool)), ("measured only", meas)):
            base = score(X0, mask)
            for name, cols in [*[(NAMES[f], [f]) for f in FEATURES], *GROUPS.items()]:
                idx = [FEATURES.index(c) for c in cols]
                drops = []
                for r in range(REPEATS):
                    X = X0.copy()
                    perm = rng.permutation(len(X))
                    X[:, idx] = X0[perm][:, idx]          # shuffle the group jointly (rows stay intact within it)
                    drops.append(base - score(X, mask))
                rows.append({"model": model, "fold": k, "cells": subset, "input": name,
                             "kind": "",
                             "baseline_bal_acc": base, "drop_mean": np.mean(drops)})
perm_imp = pd.DataFrame(rows)
perm_imp["kind"] = np.where(perm_imp.input.isin(GROUPS), "group", "single")
perm_imp.to_csv(AGG / "permutation_importance_by_fold.csv", index=False)
summ = (perm_imp.groupby(["cells", "kind", "model", "input"]).drop_mean.agg(["mean", "std"]).round(4).reset_index())
summ.to_csv(AGG / "permutation_importance_summary.csv", index=False)
display(summ.pivot_table(index=["cells", "kind", "input"], columns="model", values="mean").round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), dpi=150)
for ax, kind, title in zip(axes, ["single", "group"], ["Single inputs", "Grouped: location vs elevation"]):
    d = summ[(summ.cells == "all cells") & (summ.kind == kind)]
    inputs = list(dict.fromkeys(d.input))
    w = 0.8 / len(models)
    for j, model in enumerate(models):
        dd = d[d.model == model].set_index("input").reindex(inputs)
        ax.barh(np.arange(len(inputs)) + (j - (len(models) - 1) / 2) * w, dd["mean"], xerr=dd["std"], height=w * 0.9,
                color=plots.MODEL_COLOURS.get(model), label=model, error_kw={"linewidth": 0.8})
    ax.set_yticks(range(len(inputs)), inputs); ax.invert_yaxis()
    ax.axvline(0, color=plots.INK2, linewidth=0.8)
    ax.set_xlabel("Drop in balanced accuracy when shuffled (outer test folds)")
    ax.set_title(title, loc="left", fontsize=10); plots.style(ax, "x")
axes[1].legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(AGG / "fig_permutation_importance.png", bbox_inches="tight")

**Reading it.** A drop near 0 means the model does not rely on that input for skill at unseen locations; a negative drop means the input *hurts* at unseen locations (the model learned a local pattern that does not transfer). Compare location with elevation: if elevation dominates, the skill is mostly the depth trend; whatever location adds is the value of the horizontal information.

## 2. SHAP values (exact, model-agnostic)

In [ ]:
import shap
N_EXPLAIN = 60 if MODE == "smoke" else 300     # test cells explained per fold
N_BACKGROUND = 50 if MODE == "smoke" else 100  # training cells used as the background distribution
parts = []
for model in models:
    for k in folds_k:
        m, thr, train, test = load_fold(model, k)
        r = np.random.default_rng(seed=[cfg["seeds"]["global"], k])
        bg = train[FEATURES].values[r.choice(len(train), min(N_BACKGROUND, len(train)), replace=False)]
        ex = test.iloc[r.choice(len(test), min(N_EXPLAIN, len(test)), replace=False)]
        f = lambda X: m.predict_proba(np.asarray(X))[:, 1]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            sv = shap.explainers.Exact(f, shap.maskers.Independent(bg, max_samples=len(bg)))(ex[FEATURES].values)
        parts.append(ex[["BH_Code", "cell", "measured", *FEATURES]].assign(
            model=model, fold=k, base=sv.base_values,
            **{f"shap_{NAMES[c]}": sv.values[:, i] for i, c in enumerate(FEATURES)}))
shap_df = pd.concat(parts, ignore_index=True)
shap_df.to_csv(RES / "shap_values_by_cell.csv", index=False)
cols = [f"shap_{NAMES[c]}" for c in FEATURES]
glob_ = shap_df.groupby("model")[cols].apply(lambda d: d.abs().mean()).round(4)
glob_.columns = [c.replace("shap_", "") for c in glob_.columns]
glob_.to_csv(AGG / "shap_mean_abs_by_model.csv"); display(glob_)

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(3.2 * len(models), 3.2), dpi=150, sharex=True, squeeze=False)
for ax, model in zip(axes[0], models):
    v = glob_.loc[model].sort_values()
    ax.barh(v.index, v.values, color=plots.MODEL_COLOURS.get(model))
    ax.set_title(model, loc="left", fontsize=10); ax.set_xlabel("mean |SHAP| on P(Clay)"); plots.style(ax, "x")
fig.suptitle("Global SHAP importance (held-out cells)", x=0.01, ha="left", fontsize=10)
fig.tight_layout(); fig.savefig(AGG / "fig_shap_global.png", bbox_inches="tight")

In [ ]:
# Beeswarm: one dot per explained cell; colour = the input's value (low -> high, scaled within the input)
fig, axes = plt.subplots(1, len(models), figsize=(3.6 * len(models), 3.4), dpi=150, sharex=True, squeeze=False)
cmap = plt.get_cmap("coolwarm")
for ax, model in zip(axes[0], models):
    d = shap_df[shap_df.model == model]
    order = glob_.loc[model].sort_values().index
    for i, name in enumerate(order):
        feat = [c for c in FEATURES if NAMES[c] == name][0]
        x = d[f"shap_{name}"].values
        v = d[feat].values
        c = (v - v.min()) / (np.ptp(v) or 1)
        jitter = np.random.default_rng(i).normal(0, 0.08, len(x))
        ax.scatter(x, i + jitter, c=cmap(c), s=5, alpha=0.7, linewidth=0)
    ax.axvline(0, color=plots.INK2, linewidth=0.8)
    ax.set_yticks(range(len(order)), order); ax.set_xlabel("SHAP value (+ = towards Clay)")
    ax.set_title(model, loc="left", fontsize=10); plots.style(ax, "x")
sm = plt.cm.ScalarMappable(cmap=cmap); sm.set_array([])
fig.colorbar(sm, ax=axes[0].tolist(), shrink=0.8, ticks=[0, 1]).ax.set_yticklabels(["low", "high"])
fig.savefig(AGG / "fig_shap_beeswarm.png", bbox_inches="tight")

### Elevation dependence: does the model follow the soil profile?

The observed clay share rises with elevation up to about 10–12 m and falls again near the ground surface (sandy fill and topsoil). If a model has learned this geologically, its SHAP value for elevation follows the same shape. This is the geotechnical plausibility check.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.2), dpi=150)
for model in models:
    d = shap_df[shap_df.model == model]
    b = pd.cut(d.Top_Elevation_m, 12)
    t = d.groupby(b, observed=True).agg(z=("Top_Elevation_m", "mean"), s=("shap_Elevation", "mean"))
    ax.plot(t.z, t.s, marker="o", markersize=3, linewidth=1.4, color=plots.MODEL_COLOURS.get(model), label=model)
ax.axhline(0, color=plots.INK2, linewidth=0.8)
ax.set_xlabel("Top elevation of the cell (m)"); ax.set_ylabel("Mean SHAP of elevation (+ = towards Clay)")
ax.legend(frameon=False, fontsize=8); ax.set_title("Elevation dependence (held-out cells)", loc="left", fontsize=10)
plots.style(ax); fig.savefig(AGG / "fig_shap_elevation_dependence.png", bbox_inches="tight")
obs = mt[mt.in_A & (mt.measured == 1)]
display(obs.groupby(pd.cut(obs.Top_Elevation_m, [-10, 0, 2, 4, 6, 8, 10, 12, 14, 30]), observed=True).y
        .agg(cells="size", clay_share="mean").round(3))

**For the thesis.** Report the grouped permutation importance (location vs elevation) next to the gain over the depth-only reference in notebook 04: both answer what the horizontal coordinates add. Use the elevation-dependence plot to show whether the models reproduce the observed clay-share profile, which is the link between the machine-learning result and the site's geology.